# Lab 06 - Linear Regression: PM2.5 Prediction

This notebook applies **Lab 6 - Linear Regression** to predict hourly `PM2_5_ug_m3` values from other pollutant, city, and time features.


## Lab 6 concepts used

- Multiple linear regression.
- Polynomial feature expansion for non-linear relationships.
- Ridge, Lasso, and Elastic Net regularization.
- Regression evaluation with MAE, RMSE, and R2.

The target `PM2_5_ug_m3` is removed from the input features. `European_AQI` and `Hazardous_Event` are also excluded to reduce target-derived leakage.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

REGRESSION_NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'Carbon_Monoxide_ug_m3',
    'Nitrogen_Dioxide_ug_m3', 'Ozone_ug_m3', 'Dust_ug_m3',
    'UV_Index', 'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, REGRESSION_NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures

model_df = latest_rows(add_time_features(data), 120000)
train_df, test_df = chronological_split(model_df, train_size=0.8)
X_train = train_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['PM2_5_ug_m3']
X_test = test_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['PM2_5_ug_m3']


In [ ]:
def evaluate_regression(name, estimator):
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    return {
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': root_mean_squared_error(y_test, pred),
        'R2': r2_score(y_test, pred),
    }, pred

models = {
    'Linear Regression': Pipeline(steps=[('preprocess', preprocessor), ('model', LinearRegression())]),
    'Ridge': Pipeline(steps=[('preprocess', preprocessor), ('model', Ridge(alpha=1.0))]),
    'Lasso': Pipeline(steps=[('preprocess', preprocessor), ('model', Lasso(alpha=0.005, max_iter=10000))]),
    'Elastic Net': Pipeline(steps=[('preprocess', preprocessor), ('model', ElasticNet(alpha=0.005, l1_ratio=0.4, max_iter=10000))]),
}

results = []
predictions = {}
for name, estimator in models.items():
    result, pred = evaluate_regression(name, estimator)
    results.append(result)
    predictions[name] = pred

results_df = pd.DataFrame(results).sort_values('RMSE')
results_df


In [ ]:
poly_numeric_features = [
    'PM10_ug_m3', 'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index'
]
poly_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
])
poly_model = Pipeline(steps=[
    ('preprocess', ColumnTransformer([('num', poly_preprocessor, poly_numeric_features)], remainder='drop')),
    ('model', Ridge(alpha=5.0))
])

poly_result, poly_pred = evaluate_regression('Polynomial Ridge degree 2', poly_model)
all_results = pd.concat([results_df, pd.DataFrame([poly_result])], ignore_index=True).sort_values('RMSE')
all_results


In [ ]:
best_name = all_results.iloc[0]['model']
best_pred = poly_pred if best_name == 'Polynomial Ridge degree 2' else predictions[best_name]
plot_df = pd.DataFrame({'actual': y_test, 'predicted': best_pred}).sample(n=min(5000, len(y_test)), random_state=42)

plt.figure(figsize=(6, 5))
sns.scatterplot(data=plot_df, x='actual', y='predicted', alpha=0.25)
plt.xlabel('Actual PM2.5')
plt.ylabel('Predicted PM2.5')
plt.title(f'{best_name}: actual vs predicted')
plt.show()


## What was learned from Lab 6

Linear models create a transparent PM2.5 regression baseline. Regularization is useful for controlling correlated pollutant features, and polynomial expansion can test whether simple non-linear interactions improve results.
